# 第一周结束练习

建立在 OpenAI 聊天完成 API 之上的技术问答工具。

**此笔记本演示的内容：**

- **有状态对话** - 一个积累消息历史记录的“Chat”类，因此模型在每个回合都有完整的上下文
- **流式响应**
- **多型号比较**
- **对话实用程序** — 历史记录检查和重置以重新开始

In [ ]:
# 进口

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
# 常量

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# 设置环境

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

## 助手

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def display_md(text):
    """Render text as Markdown in the notebook."""
    display(Markdown(text))

## 聊天类 — 有状态的对话包装器

OpenAI Chat Completions API 是无状态的——它没有之前调用的记忆。  
这个“Chat”类通过维护“history”列表来模拟多轮对话  
随着每次交换而增长。每个 API 调用都会发送完整的历史记录，以便模型  
可以参考之前的上下文，就像真实的对话一样。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
class Chat:
    """Stateful chat wrapper around OpenAI's completions API."""

    def __init__(self, client, system="You are a helpful technical assistant.", model=MODEL_GPT):
        self.client = client
        self.model = model
        self.system = system
        self.history = []  # stores user + assistant messages

    def ask(self, question, show=True):
        """Send a question and get a complete response."""
        self.history.append({"role": "user", "content": question})
        messages = [{"role": "system", "content": self.system}] + self.history
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages
        )
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        if show:
            display_md(reply)
        return reply

    def ask_stream(self, question):
        """Send a question and stream the response token-by-token."""
        self.history.append({"role": "user", "content": question})
        messages = [{"role": "system", "content": self.system}] + self.history
        stream = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            stream=True
        )
        chunks = []
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end='', flush=True)
                chunks.append(delta)
        print()
        reply = ''.join(chunks)
        self.history.append({"role": "assistant", "content": reply})
        display_md(reply)

    def show_history(self):
        """Display the full conversation history."""
        display_md(f'**[SYSTEM]**\n\n{self.system}\n\n---')
        for msg in self.history:
            role = msg["role"].upper()
            display_md(f'**[{role}]**\n\n{msg["content"]}\n\n---')

    def reset(self):
        """Clear conversation history to start fresh."""
        self.history = []

## 询问 GPT-4o-mini（流媒体）

In [ ]:
# 这是问题；输入此内容以询问新问题

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
gpt_chat = Chat(openai, model=MODEL_GPT)
gpt_chat.ask_stream(question)

## 询问 Llama 3.2（通过 Ollama）

Ollama 在本地公开了兼容 OpenAI 的 API，因此我们可以重用相同的“Chat”类  
使用不同的客户端和模型 - 无需更改代码。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
llama_chat = Chat(ollama_client, model=MODEL_LLAMA)
llama_chat.ask(question)

## 查看历史记录

In [ ]:
gpt_chat.show_history()

＃＃ 重置

In [ ]:
gpt_chat.reset()

## 询问 Gemini（通过 OpenRouter）

OpenRouter 还提供了 OpenAI 兼容的端点，因此相同的 `Chat` 类  
与 Gemini 无缝协作 — 只需交换客户端和模型字符串。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
GEMINI_MODEL = 'google/gemini-3-flash-preview'

gemini_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1/"
)
print("Gemini client initialized successfully.")

gemini_chat = Chat(gemini_client, model=GEMINI_MODEL)

In [ ]:
gemini_chat.ask(question)